In [68]:
# imports
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import lightgbm as lgbm
import xgboost as xgb

In [69]:
# Last inn data
purchase_orders = pd.read_csv("data/kernel/purchase_orders.csv", parse_dates=["delivery_date", "created_date_time", "modified_date_time"]).copy()
receivals = pd.read_csv("data/kernel/receivals.csv", parse_dates=["date_arrival"]).copy()
materials = pd.read_csv("data/extended/materials.csv").copy()
transportation = pd.read_csv("data/extended/transportation.csv").copy()
mapping = pd.read_csv("data/prediction_mapping.csv", parse_dates=["forecast_start_date", "forecast_end_date"]).copy()

receivals['date_arrival'] = pd.to_datetime(receivals['date_arrival'], utc=True).dt.tz_localize(None)

# Drop data før 2007, under covid og 29 feb 2024
purchase_orders = purchase_orders[purchase_orders["created_date_time"].dt.year > 2016]
receivals = receivals[receivals["date_arrival"].dt.year > 2016]

""" receivals = receivals[~receivals["date_arrival"].dt.year.isin([2020,2021])]
purchase_orders = purchase_orders[~purchase_orders["created_date_time"].dt.year.isin([2020,2021])] """

receivals = receivals[~((receivals["date_arrival"].dt.month == 2) & (receivals["date_arrival"].dt.day == 29) & (receivals["date_arrival"].dt.year == 2024))]
purchase_orders = purchase_orders[~((purchase_orders["created_date_time"].dt.month == 2) & (purchase_orders["created_date_time"].dt.day == 29) & (purchase_orders["created_date_time"].dt.year == 2024))]

# Fjerner negative og slettede
purchase_orders_clean = purchase_orders[purchase_orders['quantity'] > 0]
purchase_orders_clean = purchase_orders_clean[purchase_orders_clean['status'] != 'Deleted']
receivals_clean = receivals[receivals['net_weight'] > 0]


# Bygger grunnlagstabeller

purchase_base = purchase_orders_clean[["purchase_order_id", 
                                       "created_date_time",
                                       "purchase_order_item_no",
                                       ]]

receivals_base = receivals_clean[["rm_id",
                                 "purchase_order_id",
                                 "purchase_order_item_no",
                                 "product_id",
                                 "date_arrival",
                                 "net_weight"
                                 ]]

# Slår sammen for å finne leveringsgrad
rec_ord_merged = receivals_base.merge(
    purchase_base,
    on = ["purchase_order_id", "purchase_order_item_no"],
    how = "inner"
    )


data = rec_ord_merged

count_rm_2024 = data.loc[data["created_date_time"].dt.year == 2024, "rm_id"].nunique()
print(f"Unique rm_id in 2024: {count_rm_2024}")

print(data.info())
data


Unique rm_id in 2024: 58
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49181 entries, 0 to 49180
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   rm_id                   49181 non-null  float64            
 1   purchase_order_id       49181 non-null  float64            
 2   purchase_order_item_no  49181 non-null  float64            
 3   product_id              49181 non-null  float64            
 4   date_arrival            49181 non-null  datetime64[ns]     
 5   net_weight              49181 non-null  float64            
 6   created_date_time       49181 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), datetime64[ns](1), float64(5)
memory usage: 2.6 MB
None


,rm_id,purchase_order_id,purchase_order_item_no,product_id,date_arrival,net_weight,created_date_time
0,2132.0,284465.0,10.0,91900330.0,2017-01-04 10:23:00,580.0,2017-01-04 12:52:00+00:00
1,2130.0,284431.0,10.0,91900143.0,2017-01-04 15:21:00,22462.0,2017-01-03 08:16:58+00:00
2,2130.0,284536.0,10.0,91900143.0,2017-01-05 12:06:00,20936.0,2017-01-10 08:30:17+00:00
3,2130.0,284431.0,10.0,91900143.0,2017-01-09 10:54:00,13676.0,2017-01-03 08:16:58+00:00
4,2131.0,284431.0,20.0,91900302.0,2017-01-09 15:32:00,7011.0,2017-01-03 08:16:58+00:00
...,...,...,...,...,...,...,...
49176,3421.0,328740.0,30.0,91900160.0,2024-12-19 10:26:00,1220.0,2024-08-22 07:50:43+00:00
49177,2145.0,328740.0,20.0,91900146.0,2024-12-19 10:26:00,850.0,2024-08-22 07:50:42+00:00
49178,3865.0,328740.0,10.0,91900143.0,2024-12-19 10:26:00,14170.0,2024-08-22 07:50:42+00:00
49179,3901.0,330334.0,10.0,91901440.0,2024-12-19 10:54:00,15020.0,2024-11-15 12:11:38+00:00


In [70]:
""" # Fjerner stock_location 'DELETED' i materials
materials['product_id'] = pd.to_numeric(materials['product_id'], errors='coerce')
materials['rm_id'] = pd.to_numeric(materials['rm_id'], errors='coerce')

active_materials = materials[~materials['stock_location'].astype(str).str.upper().str.contains('DELETED')].copy()

# 2) Tillatte par etter filtrering
allowed_pairs = (
    active_materials[['product_id', 'rm_id']]
    .dropna()
    .drop_duplicates()
)

# 3) Harmoniser typer i data og filtrer med inner merge for å beholde KUN tillatte par
data['product_id'] = pd.to_numeric(data['product_id'], errors='coerce')
data['rm_id'] = pd.to_numeric(data['rm_id'], errors='coerce')

n_before = len(data)
data = data.merge(allowed_pairs.assign(_keep=1), on=['product_id', 'rm_id'], how='inner')
removed_rows = n_before - len(data)

print(f"Kept only non-DELETED pairs: removed {removed_rows} rows (from {n_before} to {len(data)}).") """

' # Fjerner stock_location \'DELETED\' i materials\nmaterials[\'product_id\'] = pd.to_numeric(materials[\'product_id\'], errors=\'coerce\')\nmaterials[\'rm_id\'] = pd.to_numeric(materials[\'rm_id\'], errors=\'coerce\')\n\nactive_materials = materials[~materials[\'stock_location\'].astype(str).str.upper().str.contains(\'DELETED\')].copy()\n\n# 2) Tillatte par etter filtrering\nallowed_pairs = (\n    active_materials[[\'product_id\', \'rm_id\']]\n    .dropna()\n    .drop_duplicates()\n)\n\n# 3) Harmoniser typer i data og filtrer med inner merge for å beholde KUN tillatte par\ndata[\'product_id\'] = pd.to_numeric(data[\'product_id\'], errors=\'coerce\')\ndata[\'rm_id\'] = pd.to_numeric(data[\'rm_id\'], errors=\'coerce\')\n\nn_before = len(data)\ndata = data.merge(allowed_pairs.assign(_keep=1), on=[\'product_id\', \'rm_id\'], how=\'inner\')\nremoved_rows = n_before - len(data)\n\nprint(f"Kept only non-DELETED pairs: removed {removed_rows} rows (from {n_before} to {len(data)}).") '

In [71]:
df = data.copy()

for c in ["date_arrival", "created_date_time"]:
    df[c] = pd.to_datetime(df[c], errors="coerce", utc=True).dt.tz_localize(None)
    
df["lead_time_days"] = (df["date_arrival"] - df["created_date_time"]).dt.days
df["lead_time_days"] = df["lead_time_days"].clip(lower=0)
df["doy_norm"] = df["date_arrival"].dt.dayofyear / 365.0
df["doy_norm_created"] = df["created_date_time"].dt.dayofyear / 365.0

# Numerisk tidsstempel for modellinput
df["created_timestamp"] = df["created_date_time"].view("int64") // 10**9
df.loc[df["created_date_time"].isna(), "created_timestamp"] = np.nan
df["arrival_timestamp"] = df["date_arrival"].view("int64") // 10**9
df.loc[df["date_arrival"].isna(), "arrival_timestamp"] = np.nan


print(df.info())

print(df["lead_time_days"].describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49181 entries, 0 to 49180
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   rm_id                   49181 non-null  float64       
 1   purchase_order_id       49181 non-null  float64       
 2   purchase_order_item_no  49181 non-null  float64       
 3   product_id              49181 non-null  float64       
 4   date_arrival            49181 non-null  datetime64[ns]
 5   net_weight              49181 non-null  float64       
 6   created_date_time       49181 non-null  datetime64[ns]
 7   lead_time_days          49181 non-null  int64         
 8   doy_norm                49181 non-null  float64       
 9   doy_norm_created        49181 non-null  float64       
 10  created_timestamp       49181 non-null  float64       
 11  arrival_timestamp       49181 non-null  float64       
dtypes: datetime64[ns](2), float64(9), int64(1)
mem

C:\Users\strom\AppData\Local\Temp\ipykernel_18092\3935362312.py:12: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  df["created_timestamp"] = df["created_date_time"].view("int64") // 10**9
C:\Users\strom\AppData\Local\Temp\ipykernel_18092\3935362312.py:14: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  df["arrival_timestamp"] = df["date_arrival"].view("int64") // 10**9


In [72]:
# Sortér data riktig for rolling features
df_sorted = df.sort_values(['rm_id', 'date_arrival']).copy()

# Sett date_arrival som index midlertidig for rolling beregninger
df_sorted = df_sorted.set_index('date_arrival')

# Beregn rolling features for hver rm_id
rolling_features = []

for rm_id in df_sorted['rm_id'].unique():
    rm_data = df_sorted[df_sorted['rm_id'] == rm_id].copy()
    
    # 90-dagers sum
    rm_data['sum_past_90'] = rm_data['net_weight'].rolling('90D').sum()
    
    # 180-dagers sum  
    rm_data['sum_past_180'] = rm_data['net_weight'].rolling('180D').sum()
    
    rolling_features.append(rm_data)

# Kombiner alle rm_id tilbake
df_with_rolling = pd.concat(rolling_features, ignore_index=False)
df_with_rolling = df_with_rolling.reset_index()

# Forhold og differanser mellom vinduer
df_with_rolling['ratio_90_180'] = df_with_rolling['sum_past_90'] / (df_with_rolling['sum_past_180'] + 1e-8)
df_with_rolling['diff_90_180'] = df_with_rolling['sum_past_180'] - df_with_rolling['sum_past_90']

# Ukedag og helgeeffekter
df_with_rolling['weekday'] = (df_with_rolling['date_arrival'].dt.weekday)/7
df_with_rolling['is_weekend'] = (df_with_rolling['weekday'] >= 5).astype(int)

# Kontinuerlig årssyklus (sin/cos for å fange rytme)
df_with_rolling['doy_sin'] = np.sin(2 * np.pi * df_with_rolling['date_arrival'].dt.dayofyear / 365)
df_with_rolling['doy_cos'] = np.cos(2 * np.pi * df_with_rolling['date_arrival'].dt.dayofyear / 365)

# Måned som kategorisk
df_with_rolling['month'] = df_with_rolling['date_arrival'].dt.month

# Oppdater df med nye features
df = df_with_rolling.copy()

# Fyll NaN verdier i rolling features med 0 (eller forrige verdi)
df['sum_past_90'] = df['sum_past_90'].fillna(0)
df['sum_past_180'] = df['sum_past_180'].fillna(0)
df['ratio_90_180'] = df['ratio_90_180'].fillna(0)
df['diff_90_180'] = df['diff_90_180'].fillna(0)

features = [
    "rm_id",
    "lead_time_days",
    "sum_past_90",
    "sum_past_180",
    "doy_sin",
    "doy_cos",
    "month",
    "weekday",
    "is_weekend",
    "ratio_90_180",
    "diff_90_180"
    
]

x = df[features].copy()
y = df["net_weight"].values

print(f"Features: {features}")
print(f"Antall features: {len(features)}")
print(f"Shape: {x.shape}")
print(f"Nullverdier i target: {(y == 0).mean():.1%}")
print(f"Rolling features beregnet for {df['rm_id'].nunique()} unike rm_id")

Features: ['rm_id', 'lead_time_days', 'sum_past_90', 'sum_past_180', 'doy_sin', 'doy_cos', 'month', 'weekday', 'is_weekend', 'ratio_90_180', 'diff_90_180']
Antall features: 11
Shape: (49181, 11)
Nullverdier i target: 0.0%
Rolling features beregnet for 107 unike rm_id


In [73]:
# Split data: kun 2018-2024 for trening som Hannah nevnte
train_mask = (df["created_date_time"] >= pd.Timestamp("2018-01-01")) & (df["created_date_time"] < pd.Timestamp("2024-01-01"))
test_mask = df["created_date_time"] >= pd.Timestamp("2024-01-01")

X_train, Y_train = x[train_mask], y[train_mask]
X_test,  Y_test  = x[test_mask], y[test_mask]

y_train_log = np.log1p(Y_train)

print("Train:", X_train.shape, " Test:", X_test.shape)
print(f"Treningsperiode: 2018-2024")
print(f"Testperiode: 2024+")
print(f"Unique rm_id in test: {X_test['rm_id'].nunique()}")
print(f"Nullverdier i trening: {(Y_train == 0).mean():.1%}")
print(f"Nullverdier i test: {(Y_test == 0).mean():.1%}")

# Behold ID-informasjonen også
test = df.loc[test_mask].copy()

Train: (36229, 11)  Test: (5814, 11)
Treningsperiode: 2018-2024
Testperiode: 2024+
Unique rm_id in test: 58
Nullverdier i trening: 0.0%
Nullverdier i test: 0.0%


In [74]:
# 3 modeller som Hannah nevnte: LightGBM QuantileRegressor, LightGBM MAE og Sklearn QuantileRegressor
from sklearn.ensemble import GradientBoostingRegressor

# Modell 1: LightGBM QuantileRegressor med tau=0.3 (som Hannah nevnte)
model_lgbm_quantile = lgbm.LGBMRegressor(
    objective='quantile',
    alpha=0.22, 
    n_estimators=10000,
    max_depth=6,
    random_state=42,
    learning_rate=0.05,
    verbose=-1
)

# Modell 2: LightGBM MAE
model_lgbm_mae = lgbm.LGBMRegressor(
    objective='mae',
    n_estimators=10000,
    max_depth=6,
    random_state=42,
    learning_rate=0.05,
    verbose=-1
)

# Tren alle tre modeller
print("Trener LightGBM Quantile (tau=0.3)...")
model_lgbm_quantile.fit(X_train, y_train_log)
y_pred_lgbm_quantile = np.expm1(model_lgbm_quantile.predict(X_test) * 0.9)

print("Trener LightGBM MAE...")
model_lgbm_mae.fit(X_train, y_train_log)
y_pred_lgbm_mae = np.expm1(model_lgbm_mae.predict(X_test) * 0.89)

# Evaluer alle modeller
def quantile_loss(y_true, y_pred, q, rm_ids=None):
    
    if rm_ids is None:
        # Standard quantile loss
        diff = y_true - y_pred
        return np.mean(np.maximum(q * diff, (q - 1) * diff))
    
    # Kumulativ-bevisst quantile loss
    df = pd.DataFrame({
        'y_true': y_true,
        'y_pred': y_pred,
        'rm_id': rm_ids
    }).sort_values('rm_id')
    
    total_loss = 0
    total_weight = 0
    
    for rm_id in df['rm_id'].unique():
        mask = df['rm_id'] == rm_id
        y_t = df.loc[mask, 'y_true'].values
        y_p = df.loc[mask, 'y_pred'].values
        
        if len(y_t) > 1:
            # For kumulative data: beregn loss på differanser (daglige endringer)
            y_t_diff = np.diff(np.concatenate([[0], y_t]))
            y_p_diff = np.diff(np.concatenate([[0], y_p]))
            
            diff = y_t_diff - y_p_diff
            rm_loss = np.mean(np.maximum(q * diff, (q - 1) * diff))
            
            total_loss += rm_loss * len(y_t_diff)
            total_weight += len(y_t_diff)
        else:
            # Enkelt punkt: standard quantile loss
            diff = y_t - y_p
            rm_loss = np.maximum(q * diff, (q - 1) * diff)[0]
            total_loss += rm_loss
            total_weight += 1
    
    return total_loss / total_weight if total_weight > 0 else 0

# Test alle modeller med kumulative quantile loss
test_rm_ids = X_test['rm_id'].values if 'rm_id' in X_test.columns else None

print("\n=== Modellsammenligning ===")
models = {
    'LightGBM Quantile (tau=0.3)': y_pred_lgbm_quantile,
    'LightGBM MAE': y_pred_lgbm_mae,
    #'XGBoost': y_pred_xgb
}

for name, y_pred in models.items():
    rmse = np.sqrt(mean_squared_error(Y_test, y_pred))
    qloss_standard = quantile_loss(Y_test, y_pred, 0.2)
    qloss_cumulative = quantile_loss(Y_test, y_pred, 0.2, test_rm_ids)
    
    print(f"\n{name}:")
    print(f"  RMSE: {rmse:.0f}")
    print(f"  Standard QuantileLoss (q=0.2): {qloss_standard:.4f}")
    print(f"  Kumulativ QuantileLoss (q=0.2): {qloss_cumulative:.4f}")

# Velg beste modell for videre bruk
y_pred = y_pred_lgbm_quantile  # Kan endres basert på resultater
model = model_lgbm_quantile
print(f"\nBruker LightGBM Quantile (tau=0.3) for videre analyse")

Trener LightGBM Quantile (tau=0.3)...
Trener LightGBM MAE...
Trener LightGBM MAE...

=== Modellsammenligning ===

LightGBM Quantile (tau=0.3):
  RMSE: 12250
  Standard QuantileLoss (q=0.2): 2080.8893
  Kumulativ QuantileLoss (q=0.2): 2444.5881

LightGBM MAE:
  RMSE: 11901
  Standard QuantileLoss (q=0.2): 2026.6319
  Kumulativ QuantileLoss (q=0.2): 2421.6672

Bruker LightGBM Quantile (tau=0.3) for videre analyse

=== Modellsammenligning ===

LightGBM Quantile (tau=0.3):
  RMSE: 12250
  Standard QuantileLoss (q=0.2): 2080.8893
  Kumulativ QuantileLoss (q=0.2): 2444.5881

LightGBM MAE:
  RMSE: 11901
  Standard QuantileLoss (q=0.2): 2026.6319
  Kumulativ QuantileLoss (q=0.2): 2421.6672

Bruker LightGBM Quantile (tau=0.3) for videre analyse


In [75]:
# Sorter prediksjoner etter rm_id
import numpy as np
import pandas as pd

# Lag en tabell med rm_id, prediksjon og fasit
pred_df = pd.DataFrame({
    "rm_id": X_test["rm_id"].values,
    "y_pred": np.asarray(y_pred),
    "y_true": np.asarray(Y_test),
})

# Sortering etter rm_id
pred_df_sorted = pred_df.sort_values("rm_id", kind="mergesort").reset_index(drop=True)

print(pred_df_sorted.head(10))
print(f"Unique rm_id count: {pred_df_sorted['rm_id'].nunique()}")


    rm_id       y_pred   y_true
0  2123.0  2552.537391  12500.0
1  2124.0  1004.329368   4320.0
2  2124.0  1239.236163   3240.0
3  2124.0  1887.806109   8640.0
4  2125.0  1237.580282  25000.0
5  2125.0  3342.617814  25000.0
6  2125.0  2681.210384  10211.0
7  2129.0  3440.102038  14920.0
8  2129.0  3214.197103  11180.0
9  2129.0  3621.209118  11300.0
Unique rm_id count: 58


In [76]:
# Evaluering av modellen
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# Forutsetter at y_pred, Y_test, Y_train, model, X_train, X_test finnes fra forrige celle

# 1. Grunnmetrikker
mae = mean_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)

# 2. Kumulativ kvantil-tap (q=0.2) – forbedret for kumulative prognoser
def quantile_loss(y_true, y_hat, q, rm_ids=None):
    """
    Quantile loss som tar hensyn til kumulative prognoser.
    Hvis rm_ids oppgis, grupperes data og loss beregnes på daglige differanser.
    """
    if rm_ids is None:
        # Standard quantile loss
        diff = y_true - y_hat
        return np.mean(np.maximum(q * diff, (q - 1) * diff))
    
    # Kumulativ-bevisst quantile loss
    df = pd.DataFrame({
        'y_true': y_true,
        'y_hat': y_hat,
        'rm_id': rm_ids
    }).sort_values('rm_id')
    
    total_loss = 0
    total_weight = 0
    
    for rm_id in df['rm_id'].unique():
        mask = df['rm_id'] == rm_id
        y_t = df.loc[mask, 'y_true'].values
        y_p = df.loc[mask, 'y_hat'].values
        
        if len(y_t) > 1:
            # For kumulative data: beregn loss på differanser (daglige endringer)
            y_t_diff = np.diff(np.concatenate([[0], y_t]))
            y_p_diff = np.diff(np.concatenate([[0], y_p]))
            
            diff = y_t_diff - y_p_diff
            rm_loss = np.mean(np.maximum(q * diff, (q - 1) * diff))
            
            total_loss += rm_loss * len(y_t_diff)
            total_weight += len(y_t_diff)
        else:
            # Enkelt punkt: standard quantile loss
            diff = y_t - y_p
            rm_loss = np.maximum(q * diff, (q - 1) * diff)[0]
            total_loss += rm_loss
            total_weight += 1
    
    return total_loss / total_weight if total_weight > 0 else 0

# Beregn quantile loss med rm_id informasjon for kumulativ behandling
test_rm_ids = X_test['rm_id'].values if 'rm_id' in X_test.columns else None
qloss = quantile_loss(Y_test, y_pred, 0.2, test_rm_ids)

# 3. Baselines
# a) Naiv gjennomsnitt av treningsmålet
baseline_mean = np.full_like(Y_test, Y_train.mean())
baseline_mean_rmse = np.sqrt(mean_squared_error(Y_test, baseline_mean))
# b) Bruke "quantity" som proxy (hvis den finnes i feature settet)
if 'quantity' in X_test.columns:
    baseline_quantity = X_test['quantity'].values
    baseline_quantity_rmse = np.sqrt(mean_squared_error(Y_test, baseline_quantity))
else:
    baseline_quantity_rmse = np.nan

# 4. Residualanalyse
residuals = Y_test - y_pred
res_summary = {
    'residual_mean': residuals.mean(),
    'residual_std': residuals.std(),
    'abs_resid_p50': np.percentile(np.abs(residuals), 50),
    'abs_resid_p90': np.percentile(np.abs(residuals), 90),
    'abs_resid_p95': np.percentile(np.abs(residuals), 95),
    'pct_over_predicted': (residuals < 0).mean(),  # hvor ofte vi overestimerer
}

# 5. Feature importance (kun for trebasert modell)
feature_importance = None
if hasattr(model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)

# 6. Samle alt i en tabell
metrics = {
    'RMSE': float(rmse),
    'MAE': float(mae),
    'R2': float(r2),
    'QuantileLoss(q=0.2)': float(qloss),
    'BaselineMean_RMSE': float(baseline_mean_rmse),
    'BaselineQuantity_RMSE': float(baseline_quantity_rmse),
}

print("=== Modellmetrikker ===")
for k, v in metrics.items():
    print(f"{k:22s}: {v:.4f}")

print("\n=== Residualanalyse ===")
for k, v in res_summary.items():
    print(f"{k:22s}: {v:.4f}")

if feature_importance is not None:
    print("\n=== Viktigste features (topp 10) ===")
    print(feature_importance.head(10).to_string(index=False))

# 7. Enkel sanity check: Modell bør slå baseline_mean på RMSE
if rmse < baseline_mean_rmse:
    print("\nModellen slår baseline (gjennomsnitt) på RMSE.")
else:
    print("\nAdvarsel: Modellen slår ikke baseline (gjennomsnitt) – vurder flere features / tuning.")

# 8. Lag en liten DataFrame for inspeksjon
preview = pd.DataFrame({
    'y_true': Y_test[:20],
    'y_pred': y_pred[:20],
    'residual': residuals[:20]
})
print("\nEksempel (første 20 prediksjoner):")
print(preview.to_string(index=False))


=== Modellmetrikker ===
RMSE                  : 11901.0598
MAE                   : 10129.1540
R2                    : -0.8942
QuantileLoss(q=0.2)   : 2444.5881
BaselineMean_RMSE     : 8937.1965
BaselineQuantity_RMSE : nan

=== Residualanalyse ===
residual_mean         : 9945.6258
residual_std          : 7150.9506
abs_resid_p50         : 10873.3464
abs_resid_p90         : 18600.6989
abs_resid_p95         : 19728.5753
pct_over_predicted    : 0.0673

=== Viktigste features (topp 10) ===
       feature  importance
lead_time_days       42804
  ratio_90_180       30402
       doy_sin       28722
       doy_cos       27722
         rm_id       26040
   sum_past_90       25388
  sum_past_180       18967
   diff_90_180       12224
       weekday       11101
         month        3453

Advarsel: Modellen slår ikke baseline (gjennomsnitt) – vurder flere features / tuning.

Eksempel (første 20 prediksjoner):
 y_true      y_pred     residual
12500.0 2552.537391  9947.462609
 4320.0 1004.329368  331

In [77]:
# If kaggle_metric.py is in the same directory, import as a module
from kaggle_metric import score, ParticipantVisibleError

# Mapping for 2025 data
mapping = pd.read_csv("data/prediction_mapping.csv", parse_dates=["forecast_start_date", "forecast_end_date"]).copy()

# Robust kolonnenavn-håndtering
if "rm_id" not in mapping.columns:
    for alt in ["material_id", "RM_ID", "rmId"]:
        if alt in mapping.columns:
            mapping = mapping.rename(columns={alt: "rm_id"})
            break

if "ID" not in mapping.columns:
    for alt in ["id", "Id"]:
        if alt in mapping.columns:
            mapping = mapping.rename(columns={alt: "ID"})
            break
    if "ID" not in mapping.columns and "rm_id" in mapping.columns:
        # Fallback: bruk rm_id som ID hvis ID mangler
        mapping["ID"] = mapping["rm_id"].astype(int)

# Sikkerhetssjekk
required_cols = {"ID", "rm_id"}
missing_cols = required_cols - set(mapping.columns)
if missing_cols:
    raise ValueError(f"Mapping mangler kolonner: {missing_cols}. Finnes: {mapping.columns.tolist()}")

# Bygg rm_id-nivå prediksjoner fra 2024-testen
if "X_test" not in globals() or "y_pred" not in globals():
    raise RuntimeError("X_test/y_pred mangler i minnet. Kjør trenings-/prediksjonscellene først.")

preds_2024_rm = pd.DataFrame({
    "rm_id": pd.to_numeric(X_test["rm_id"], errors="coerce"),
    "predicted_weight": np.clip(np.asarray(y_pred), 0, None)
}).groupby("rm_id", as_index=False).agg(predicted_weight=("predicted_weight", "sum"))

# Valgfritt: begrens til top-N rm_id for 2025-prediksjon
TOP_N_RM_2025 = 100  # Sett f.eks. 100 for top 100, eller None for alle
if TOP_N_RM_2025 is not None and int(TOP_N_RM_2025) > 0:
    top_n = int(TOP_N_RM_2025)
    total_rm = len(preds_2024_rm)
    preds_2024_rm = (
        preds_2024_rm.sort_values("predicted_weight", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    print(f"Bruker top {len(preds_2024_rm)} rm_id (av {total_rm}) for 2025.")
else:
    print(f"Bruker alle {len(preds_2024_rm)} rm_id for 2025.")

# Harmoniser typer før join
mapping["rm_id"] = pd.to_numeric(mapping["rm_id"], errors="coerce")

# Map 2024-prediksjoner til 2025-IDer
submission_2025 = mapping[["ID", "rm_id"]].merge(preds_2024_rm, on="rm_id", how="left")
missing = int(submission_2025["predicted_weight"].isna().sum())
submission_2025["predicted_weight"] = submission_2025["predicted_weight"].fillna(0.0)
submission_2025 = submission_2025[["ID", "predicted_weight"]].sort_values("ID").reset_index(drop=True)

# Skriv totalsum til egen fil (ikke submission.csv lenger)
out_path_totals = "submission_totals.csv"
submission_2025.to_csv(out_path_totals, index=False)
print(f"submission_totals.csv skrevet: {len(submission_2025)} rader. Fylte 0 for {missing} manglende rm_id i mapping.")
print(submission_2025.head(10).to_string(index=False))

# Backtest for 2024 (beholder, men påvirker ikke daglig submission)
receivals["date_arrival"] = pd.to_datetime(receivals["date_arrival"], errors="coerce", utc=True).dt.tz_localize(None)

start = pd.Timestamp("2024-01-01")
end   = pd.Timestamp("2024-05-31 23:59:59")
solution = (receivals.loc[(receivals["date_arrival"] >= start) & (receivals["date_arrival"] <= end)]
            .groupby("rm_id", as_index=False)
            .agg(weight=("net_weight", "sum"))
           ).rename(columns={"rm_id": "ID"})

rm_col = "rm_id_raw" if "rm_id_raw" in test.columns else "rm_id"

preds = pd.DataFrame({
    "ID": test[rm_col].values,
    "predicted_weight": np.clip(y_pred, 0, None)  
})

submission = preds.groupby("ID", as_index=False).agg(predicted_weight=("predicted_weight", "sum"))

submission = solution[["ID"]].merge(submission, on="ID", how="left")
submission["predicted_weight"] = submission["predicted_weight"].fillna(0.0)

try:
    final_score = score(solution=solution, submission=submission, row_id_column_name="ID")
    print("Quantile loss (q=0.2) – backtest jan–mai 2024:", final_score)
except ParticipantVisibleError as e:
    print("Scoring feilet:", e)

print(submission.head(10))
print(solution.head(10))

Bruker top 58 rm_id (av 58) for 2025.

submission_totals.csv skrevet: 30450 rader. Fylte 0 for 21750 manglende rm_id i mapping.
 ID  predicted_weight
  1               0.0
  2               0.0
  3               0.0
  4               0.0
  5               0.0
  6               0.0
  7               0.0
  8               0.0
  9               0.0
 10               0.0
Quantile loss (q=0.2) – backtest jan–mai 2024: 79963.3587710232
       ID  predicted_weight
0  2124.0      4.131372e+03
1  2125.0      7.261408e+03
2  2129.0      4.669477e+04
3  2130.0      4.215115e+06
4  2131.0      4.885753e+04
5  2132.0      2.630102e+04
6  2133.0      6.494956e+03
7  2134.0      1.139889e+05
8  2135.0      4.721453e+04
9  2140.0      3.807997e+05
       ID     weight
0  2124.0     7560.0
1  2125.0    25000.0
2  2129.0    69980.0
3  2130.0  3542751.0
4  2131.0   230553.0
5  2132.0   162130.0
6  2133.0    38414.0
7  2134.0   612678.0
8  2135.0   493170.0
9  2140.0  1046440.0
submission_totals.csv skrev

In [78]:
# Mapping for 2025 data
mapping = pd.read_csv("data/prediction_mapping.csv", parse_dates=["forecast_start_date", "forecast_end_date"]).copy()


In [79]:


# Valg for fordeling av daglige shares
going_mode = 'historic'   # 'uniform' | 'historic' | 'blended'
blend_alpha = 0.30        # andel uniform i blend når goiong_mode == 'blended'

# Forutsetninger
if 'preds_2024_rm' not in globals():
    raise RuntimeError("Mangler 'preds_2024_rm' (total per rm_id). Kjør mapping/prediksjonscellen først.")
if 'mapping' not in globals():
    raise RuntimeError("Mangler 'mapping'. Les 'data/prediction_mapping.csv' først.")

# 1) Historiske andeler (shares) per rm_id per dag-av-året (DOY) for jan–mai
receivals_hist = receivals.copy()
receivals_hist["date_arrival"] = pd.to_datetime(receivals_hist["date_arrival"], errors="coerce", utc=True).dt.tz_localize(None)
receivals_hist = receivals_hist.loc[
    (receivals_hist["date_arrival"] < pd.Timestamp("2024-01-01")) &
    (receivals_hist["date_arrival"].dt.month.isin([1,2,3,4,5]))
].copy()
receivals_hist["doy"] = receivals_hist["date_arrival"].dt.dayofyear

by_rm_doy = (
    receivals_hist
    .groupby(["rm_id", "doy"], as_index=False)
    .agg(weight=("net_weight", "sum"))
)
by_rm_tot = by_rm_doy.groupby("rm_id", as_index=False).agg(total=("weight", "sum"))
shares_rm = by_rm_doy.merge(by_rm_tot, on="rm_id", how="left")
shares_rm["share_rm"] = shares_rm["weight"] / shares_rm["total"].replace({0: np.nan})
shares_rm = shares_rm[["rm_id", "doy", "share_rm"]]

# Globale andeler per DOY for fallback
by_doy_glob = receivals_hist.groupby("doy", as_index=False).agg(weight=("net_weight", "sum"))
if by_doy_glob["weight"].sum() <= 0 or by_doy_glob.empty:
    # hvis ingen historikk: antas lik fordeling over jan–mai 2025
    # vi fyller DOY senere basert på horisonten
    by_doy_glob = pd.DataFrame(columns=["doy", "weight"])  

# 2) Definer horisont (datoer) fra mapping
min_start = mapping["forecast_start_date"].min()
max_end   = mapping["forecast_end_date"].max()
all_days = pd.date_range(min_start, max_end, freq='D')
cal = pd.DataFrame({"date": all_days})
cal["doy"] = cal["date"].dt.dayofyear

# Sørg for globale shares for ALLE DOY i horisonten
by_doy_glob = cal[["doy"]].merge(by_doy_glob, on="doy", how="left")
by_doy_glob["weight"] = by_doy_glob["weight"].fillna(1.0)
by_doy_glob["share_glob"] = by_doy_glob["weight"] / by_doy_glob["weight"].sum()
by_doy_glob = by_doy_glob[["doy", "share_glob"]]

# 3) Lag full grid av (rm_id, doy) over hele horisonten
rm_ids = preds_2024_rm["rm_id"].dropna().unique()
rm_grid = pd.DataFrame({"rm_id": np.repeat(rm_ids, len(cal)),
                        "doy": np.tile(cal["doy"].values, len(rm_ids))})

# Knytt rm-spesifikke shares og globale shares
rm_grid = rm_grid.merge(shares_rm, on=["rm_id", "doy"], how="left")
rm_grid = rm_grid.merge(by_doy_glob, on="doy", how="left")

# Velg share: historic vs blended vs uniform
if going_mode == 'uniform':
    rm_grid["share_use"] = 1.0
elif going_mode == 'blended':
    rm_grid["share_use"] = (1.0 - blend_alpha) * rm_grid["share_rm"].fillna(0.0) + blend_alpha * 1.0
else:  # 'historic'
    rm_grid["share_use"] = rm_grid["share_rm"].fillna(rm_grid["share_glob"]).fillna(0.0)

# Normaliser shares innen hver rm_id over HELE horisonten
sum_share = rm_grid.groupby("rm_id")["share_use"].transform("sum").replace({0: np.nan})
rm_grid["share_norm"] = rm_grid["share_use"] / sum_share

# 4) Knytt rm-total og lag daglig vekt per rm_id per dag
rm_grid = rm_grid.merge(preds_2024_rm.rename(columns={"predicted_weight": "rm_total"}), on="rm_id", how="left")
rm_grid["rm_total"] = rm_grid["rm_total"].fillna(0.0)
rm_grid["daily_weight"] = rm_grid["rm_total"] * rm_grid["share_norm"].fillna(0.0)

# Knytt kalenderdato til grid for å få faktiske datoer
rm_daily = rm_grid.merge(cal, on="doy", how="left")[["rm_id", "date", "daily_weight"]]
rm_daily = rm_daily.sort_values(["rm_id", "date"]).reset_index(drop=True)
rm_daily["cumulative_weight"] = rm_daily.groupby("rm_id")["daily_weight"].cumsum()

# 5) Lag submission ved å hente kumulativ vekt for hvert ID-vindu (end - (start-1))
mp = mapping[["ID", "rm_id", "forecast_start_date", "forecast_end_date"]].copy()
mp = mp.merge(rm_daily.rename(columns={"date": "forecast_end_date", "cumulative_weight": "cum_end"}),
              on=["rm_id", "forecast_end_date"], how="left")
# For start-1 dag
mp["start_prev"] = mp["forecast_start_date"] - pd.Timedelta(days=1)
mp = mp.merge(rm_daily.rename(columns={"date": "start_prev", "cumulative_weight": "cum_prev"})[["rm_id", "start_prev", "cum_prev"]],
              on=["rm_id", "start_prev"], how="left")
mp["cum_prev"] = mp["cum_prev"].fillna(0.0)

mp["predicted_weight"] = (mp["cum_end"].fillna(0.0) - mp["cum_prev"]).clip(lower=0)
submission = mp[["ID", "predicted_weight"]].copy()

# 6) Sortér og valider mot sample_submission (én rad per ID)
sample = pd.read_csv("data/sample_submission.csv")
submission = sample[["ID"]].merge(submission, on="ID", how="left")
submission["predicted_weight"] = submission["predicted_weight"].fillna(0.0)

# Skrive kun submission.csv
submission.to_csv("submission.csv", index=False)
total_predicted_weight = submission["predicted_weight"].sum()
print(f"submission.csv skrevet: {len(submission)} rader, unike ID: {submission['ID'].nunique()}")
print(f"Total predikert vekt: {total_predicted_weight:,.0f} kg ({total_predicted_weight/1e6:.1f} millioner kg)")
print(submission.head(5).to_string(index=False))
submission_with_meta = (
    mapping[["ID", "rm_id", "forecast_end_date"]]
    .merge(submission, on="ID", how="left")
)
submission_with_meta = submission_with_meta.sort_values(["rm_id", "forecast_end_date"])
total_predicted_weight = submission_with_meta.groupby("rm_id")["predicted_weight"].last().sum()

print(f"submission.csv skrevet: {len(submission)} rader, unike ID: {submission['ID'].nunique()}")
print(
    f"Total predikert vekt (siste kumulative punkt per rm_id): "
    f"{total_predicted_weight:,.0f} kg ({total_predicted_weight/1e6:.1f} millioner kg)"
 )

submission.csv skrevet: 30450 rader, unike ID: 30450
Total predikert vekt: 1,803,893,093 kg (1803.9 millioner kg)
 ID  predicted_weight
  1               0.0
  2               0.0
  3               0.0
  4               0.0
  5               0.0
submission.csv skrevet: 30450 rader, unike ID: 30450
Total predikert vekt (siste kumulative punkt per rm_id): 24,487,766 kg (24.5 millioner kg)
